# Can we predict a warranty claim within a year?
Target: `claim_within_year`. Follows `checklist.md`.

**Split design.** `device_serial` is unique per row (checked below) and there is no date column, so there is no time order and no repeated entity. A stratified random split is therefore the appropriate structure for both the test set and the CV. `device_serial` is an identifier and is dropped. All other columns (channel, age, price, daily hours) are recorded at sale/registration, i.e. known at prediction time.

In [ ]:
import numpy as np, pandas as pd
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score, precision_score, recall_score
SEED=0
df=pd.read_csv('data.csv')
print(df.shape, 'unique serials:', df.device_serial.nunique(), 'missing:', int(df.isna().sum().sum()))
print(df.claim_within_year.mean())

## 1. Hold out the test set (before any EDA or modelling)
Serials are unique, so rows are whole entities. 20% stratified hold-out.

In [ ]:
train,test=train_test_split(df,test_size=0.2,stratify=df.claim_within_year,random_state=SEED)
FEATS=['channel','device_age_years','price_eur','daily_hours']   # device_serial dropped (identifier)
Xtr,ytr=train[FEATS],train.claim_within_year
Xte,yte=test[FEATS],test.claim_within_year
print(len(train),len(test),ytr.mean(),yte.mean())

## 2. Exploration (training rows only)

In [ ]:
print(train.describe().T)
print(train.groupby('channel').claim_within_year.agg(['mean','size']))
for f in FEATS[1:]: print(f, train.groupby(pd.qcut(train[f],5,duplicates='drop'),observed=True).claim_within_year.mean().round(3).tolist())

## 3. Candidate models and metric (declared before any scoring)
1. **Baseline**: predicts the training prevalence (`DummyClassifier(prior)`).
2. **Logistic regression** (one-hot channel, standardised numerics, C=1).
3. **Logistic regression, small C** (C=0.05).
4. **Gradient boosting** (`HistGradientBoostingClassifier`, shallow, lr 0.05, 150 iterations, no early stopping) and a **deeper** variant.

Selection metric: CV average precision (PR-AUC), since claims are ~12% positive; ROC-AUC reported too. Decision threshold is picked on out-of-fold predictions to maximise F1. Preprocessing lives inside the Pipeline, so it is fitted within each fold.

In [ ]:
pre=lambda: ColumnTransformer([('c',OneHotEncoder(handle_unknown='ignore'),['channel']),
                               ('n',StandardScaler(),FEATS[1:])])
cands={
 'baseline_prior':Pipeline([('m',DummyClassifier(strategy='prior'))]),
 'logreg_C1':Pipeline([('p',pre()),('m',LogisticRegression(C=1,max_iter=1000))]),
 'logreg_C0.05':Pipeline([('p',pre()),('m',LogisticRegression(C=0.05,max_iter=1000))]),
 'hgb_shallow':Pipeline([('p',pre()),('m',HistGradientBoostingClassifier(max_depth=3,learning_rate=0.05,max_iter=150,early_stopping=False,random_state=SEED))]),
 'hgb_deep':Pipeline([('p',pre()),('m',HistGradientBoostingClassifier(max_depth=6,learning_rate=0.05,max_iter=300,early_stopping=False,random_state=SEED))]),
}

## 4. Cross-validation on training rows only (stratified 5-fold; each entity is one row, so folds hold whole entities)

In [ ]:
cv=StratifiedKFold(5,shuffle=True,random_state=SEED)
oof={};rows=[]
for n,m in cands.items():
    p=cross_val_predict(m,Xtr,ytr,cv=cv,method='predict_proba')[:,1]; oof[n]=p
    rows.append((n,average_precision_score(ytr,p),roc_auc_score(ytr,p)))
cvres=pd.DataFrame(rows,columns=['model','cv_AP','cv_AUC']).set_index('model').round(4);cvres

## 5. Choose model and threshold on CV only

In [ ]:
best=cvres.cv_AP.idxmax()
ths=np.linspace(0.02,0.6,117)
f1s=[f1_score(ytr,oof[best]>=t) for t in ths]
thr=float(ths[int(np.argmax(f1s))])
print('chosen model:',best,'| threshold:',round(thr,3),'| CV F1:',round(max(f1s),4),
      '| CV precision/recall:',round(precision_score(ytr,oof[best]>=thr),3),round(recall_score(ytr,oof[best]>=thr),3))

**Choice (recorded before touching the test set):** the model and threshold printed above — the CV-AP winner among the candidates, with the F1-maximising out-of-fold threshold. No further tuning after this point.

## 6. Refit on all training rows; score the test set once

In [ ]:
final=cands[best].fit(Xtr,ytr)
pt=final.predict_proba(Xte)[:,1]
base=cands['baseline_prior'].fit(Xtr,ytr).predict_proba(Xte)[:,1]
pred=pt>=thr
rng=np.random.default_rng(SEED);yv=yte.values;aps=[];aucs=[]
for _ in range(1000):
    i=rng.integers(0,len(yv),len(yv))
    if yv[i].min()==yv[i].max(): continue
    aps.append(average_precision_score(yv[i],pt[i]));aucs.append(roc_auc_score(yv[i],pt[i]))
ci=lambda a:np.percentile(a,[2.5,97.5]).round(3)
print(f'HEADLINE test ({best}): AP={average_precision_score(yte,pt):.3f} {ci(aps)}  AUC={roc_auc_score(yte,pt):.3f} {ci(aucs)}')
print(f'prevalence/baseline AP={average_precision_score(yte,base):.3f}')
print(f'at threshold {thr:.3f}: precision={precision_score(yte,pred):.3f} recall={recall_score(yte,pred):.3f} F1={f1_score(yte,pred):.3f}')

## 7. Conclusion
See the headline test numbers above: compare AP with the prevalence baseline and AUC with 0.5. The bootstrap intervals show uncertainty from the finite test set. The test set was scored once; no choices were revised after seeing it.